# Grok-rl-07-continuous-sac

**Stage 07 — Continuous Control with SAC**

## 概念
机器人关节力矩是**连续动作**。离散 DQN 不适用。

**SAC (Soft Actor-Critic)**：最大熵 RL — 在优化回报的同时最大化策略熵，探索更稳，是连续控制主流之一。

## 环境
Pendulum from scratch：状态 `[cosθ, sinθ, θdot]`，动作扭矩 ∈ [-2,2]。


In [ ]:

import json, math, random, time
from collections import deque
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal

OUT=Path("/kaggle/working"); OUT.mkdir(exist_ok=True)
SEED=1
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
gpu={"cuda":torch.cuda.is_available(),"device_count":torch.cuda.device_count(),"names":[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else []}
print(gpu, device)

class PendulumEnv:
    def __init__(self):
        self.max_speed=8.0; self.max_torque=2.0; self.dt=0.05
        self.g=10.0; self.m=1.0; self.l=1.0; self.reset()
    def reset(self):
        self.state=np.array([np.random.uniform(-np.pi,np.pi), np.random.uniform(-1,1)],dtype=np.float32)
        return self._obs()
    def _obs(self):
        th,thdot=self.state
        return np.array([np.cos(th), np.sin(th), thdot],dtype=np.float32)
    def step(self,u):
        th,thdot=self.state
        u=float(np.clip(u, -self.max_torque, self.max_torque))
        costs=angle_normalize(th)**2 + 0.1*thdot**2 + 0.001*(u**2)
        newthdot=thdot + (-3*self.g/(2*self.l)*np.sin(th+np.pi) + 3.0/(self.m*self.l**2)*u)*self.dt
        newth=th+newthdot*self.dt
        newthdot=np.clip(newthdot, -self.max_speed, self.max_speed)
        self.state=np.array([newth, newthdot],dtype=np.float32)
        return self._obs(), -costs, False, {}

def angle_normalize(x):
    return ((x+np.pi)%(2*np.pi))-np.pi

class Replay:
    def __init__(self, cap=100000):
        self.b=deque(maxlen=cap)
    def add(self,*x): self.b.append(x)
    def sample(self,n):
        batch=random.sample(self.b,n)
        s,a,r,ns,d=map(np.array, zip(*batch))
        return s,a,r.astype(np.float32),ns,d.astype(np.float32)
    def __len__(self): return len(self.b)

LOG_STD_MIN, LOG_STD_MAX = -20, 2

class Actor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(3,128),nn.ReLU(),nn.Linear(128,128),nn.ReLU())
        self.mu=nn.Linear(128,1); self.log_std=nn.Linear(128,1)
    def forward(self,s):
        h=self.net(s)
        mu=self.mu(h)
        log_std=torch.clamp(self.log_std(h), LOG_STD_MIN, LOG_STD_MAX)
        return mu, log_std
    def sample(self,s):
        mu, log_std=self.forward(s)
        std=log_std.exp()
        dist=Normal(mu,std)
        x=dist.rsample()
        a=torch.tanh(x)*2.0
        # log prob with tanh correction
        logp=dist.log_prob(x) - torch.log(1-torch.tanh(x).pow(2)+1e-6)
        logp=logp.sum(-1)
        return a, logp

class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.q1=nn.Sequential(nn.Linear(4,128),nn.ReLU(),nn.Linear(128,128),nn.ReLU(),nn.Linear(128,1))
        self.q2=nn.Sequential(nn.Linear(4,128),nn.ReLU(),nn.Linear(128,128),nn.ReLU(),nn.Linear(128,1))
    def forward(self,s,a):
        x=torch.cat([s,a],-1)
        return self.q1(x).squeeze(-1), self.q2(x).squeeze(-1)


In [ ]:

def train_sac(steps=25000, start_steps=1000, batch=128, gamma=0.99, tau=0.005, lr=3e-4):
    env=PendulumEnv()
    actor=Actor().to(device)
    critic=Critic().to(device)
    critic_t=Critic().to(device); critic_t.load_state_dict(critic.state_dict())
    pi_opt=torch.optim.Adam(actor.parameters(), lr=lr)
    q_opt=torch.optim.Adam(critic.parameters(), lr=lr)
    # automatic entropy tuning
    target_entropy=-1.0
    log_alpha=torch.zeros(1, requires_grad=True, device=device)
    a_opt=torch.optim.Adam([log_alpha], lr=lr)
    rb=Replay(); s=env.reset()
    ep_ret=0.0; ep_lens=0; hist=[]
    for t in range(1, steps+1):
        if t < start_steps:
            a=np.array([np.random.uniform(-2,2)],dtype=np.float32)
        else:
            with torch.no_grad():
                a_t,_=actor.sample(torch.tensor(s,device=device).unsqueeze(0))
                a=a_t.cpu().numpy()[0]
        ns,r,done,_=env.step(a[0]); ep_ret+=r; ep_lens+=1
        rb.add(s,a,r,ns,0.0)
        s=ns
        if ep_lens>=200:
            hist.append(ep_ret); ep_ret=0.0; ep_lens=0; s=env.reset()
        if len(rb)<batch: continue
        bs,ba,br,bns,bd=rb.sample(batch)
        bs=torch.tensor(bs,device=device); ba=torch.tensor(ba,device=device)
        br=torch.tensor(br,device=device); bns=torch.tensor(bns,device=device)
        with torch.no_grad():
            na, logp_na = actor.sample(bns)
            q1t,q2t=critic_t(bns, na)
            qt=torch.min(q1t,q2t) - log_alpha.exp()*logp_na
            y=br + gamma*qt
        q1,q2=critic(bs,ba)
        q_loss=F.mse_loss(q1,y)+F.mse_loss(q2,y)
        q_opt.zero_grad(); q_loss.backward(); q_opt.step()
        na, logp = actor.sample(bs)
        q1_pi,q2_pi=critic(bs, na)
        q_pi=torch.min(q1_pi,q2_pi)
        pi_loss=(log_alpha.exp().detach()*logp - q_pi).mean()
        pi_opt.zero_grad(); pi_loss.backward(); pi_opt.step()
        alpha_loss=-(log_alpha.exp() * (logp + target_entropy).detach()).mean()
        a_opt.zero_grad(); alpha_loss.backward(); a_opt.step()
        with torch.no_grad():
            for p,pt in zip(critic.parameters(), critic_t.parameters()):
                pt.data.mul_(1-tau); pt.data.add_(tau*p.data)
    return np.array(hist), actor

t0=time.time()
hist, actor = train_sac()
elapsed=time.time()-t0
print("episodes", len(hist), "last10 mean", hist[-10:].mean() if len(hist)>=10 else hist.mean())
print("elapsed", elapsed)

# random baseline
def eval_random(n=10):
    env=PendulumEnv(); rets=[]
    for _ in range(n):
        s=env.reset(); R=0
        for _ in range(200):
            a=np.random.uniform(-2,2); s,r,_,_=env.step(a); R+=r
        rets.append(R)
    return float(np.mean(rets))

def eval_actor(n=10):
    env=PendulumEnv(); rets=[]
    actor.eval()
    for _ in range(n):
        s=env.reset(); R=0
        for _ in range(200):
            with torch.no_grad():
                mu, _ = actor.forward(torch.tensor(s,device=device).unsqueeze(0))
                a=torch.tanh(mu)*2.0
            s,r,_,_=env.step(float(a.cpu().numpy()[0,0])); R+=r
        rets.append(R)
    return float(np.mean(rets))

rand_score=eval_random(); sac_score=eval_actor()
print("random", rand_score, "sac", sac_score)


In [ ]:

fig,ax=plt.subplots(figsize=(8,4))
if len(hist)>5:
    w=5; sm=np.convolve(hist, np.ones(w)/w, mode="valid")
    ax.plot(sm, label="SAC return (smoothed)")
ax.axhline(rand_score, ls="--", c="gray", label="random eval")
ax.axhline(sac_score, ls="--", c="green", label="SAC greedy eval")
ax.legend(); ax.set_title("SAC on Pendulum (from scratch)"); ax.set_xlabel("episode"); ax.set_ylabel("return")
fig.tight_layout(); fig.savefig(OUT/"stage07_sac_pendulum.png", dpi=120); plt.close(fig)

payload={
  "ok": True,
  "stage":"07-continuous-sac",
  "title":"Grok-rl-07-continuous-sac",
  "metrics":{"random_eval": rand_score, "sac_eval": sac_score, "last10_train_mean": float(hist[-10:].mean()) if len(hist)>=10 else float(hist.mean())},
  "gpu": gpu, "elapsed_sec": elapsed,
  "concept": "max-entropy continuous actor-critic for torque control",
  "new_capability": "stable continuous action RL suitable for robot joints",
  "compare_to_previous": "Stage06 PD needed model/reference; Stage07 learns torque policy from reward only",
}
# SAC should beat random on pendulum (less negative)
assert payload["metrics"]["sac_eval"] > payload["metrics"]["random_eval"] + 50 or payload["metrics"]["last10_train_mean"] > payload["metrics"]["random_eval"]
(OUT/"results_stage07.json").write_text(json.dumps(payload, indent=2))
print(json.dumps(payload, indent=2))
print("STAGE07_OK")
